In [1]:
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u

from reproject.mosaicking import find_optimal_celestial_wcs, reproject_and_coadd
from reproject import reproject_interp

from spectral_cube import SpectralCube as sc

from tqdm.notebook import trange
import glob
import warnings
warnings.filterwarnings("ignore")

In [2]:
fits_file_list = glob.glob('/home/firestar/D/NAOC/HVC-catalog/HIPASS/*chop.fits')
hdus = [fits.open(f)[0] for f in fits_file_list]
n_chan = hdus[0].data.shape[0]
n_chan

1024

In [3]:
# 先为第0通道做WCS计算
slice_hdus = []
for hdu in hdus:
    data2d = hdu.data[0]
    wcs3d = WCS(hdu.header)
    wcs2d = wcs3d.celestial
    header2d = wcs2d.to_header()
    slice_hdus.append(fits.PrimaryHDU(data2d, header=header2d))
wcs_out, shape_out = find_optimal_celestial_wcs(slice_hdus, projection='CAR')

wcs_out.wcs.crpix[1] -= wcs_out.wcs.crval[1] / wcs_out.wcs.cdelt[1]
wcs_out.wcs.crval[1] = 0
wcs_out, shape_out

(WCS Keywords
 
 Number of WCS axes: 2
 CTYPE : 'RA---CAR' 'DEC--CAR' 
 CRVAL : 99.50374974942945 0.0 
 CRPIX : 679.7561336659564 259.3424159570403 
 PC1_1 PC1_2  : 1.0 0.0 
 PC2_1 PC2_2  : 0.0 1.0 
 CDELT : -0.0666666701436 0.0666666701436 
 NAXIS : 0  0,
 (309, 1360))

In [ ]:
# 逐通道马赛克
mosaics = []
for i in trange(144):  # -600km/s to 600km/s --> channel 52 to 143 (count from 0)
    slice_hdus = []
    for hdu in hdus:
        data2d = hdu.data[i]
        wcs3d = WCS(hdu.header)
        wcs2d = wcs3d.celestial
        header2d = wcs2d.to_header()
        slice_hdus.append(fits.PrimaryHDU(data2d, header=header2d))
    array, footprint = reproject_and_coadd(
        slice_hdus,
        wcs_out,
        shape_out=shape_out,
        reproject_function=reproject_interp
    )
    mosaics.append(array)

mosaic_cube = np.stack(mosaics, axis=0)

  0%|          | 0/145 [00:00<?, ?it/s]

In [ ]:
# 假设你有原始 cube 的 header 和新的 wcs_out
new_header = hdu.header.copy()  # hdu_cube 是你的原始cube HDU
header_out = wcs_out.to_header()        # wcs_out 是新的2D WCS对象

# 覆盖原header的第一、第二坐标轴相关关键字
for key in header_out:
    if key[-1] in ('1', '2'):  # 只替换结尾为1或2的关键字
        new_header[key] = header_out[key]

# new_header 现在就是你要用的新header

new_header['NAXIS3'] = 196
#new_header['CRPIX3'] = 22
#new_header['ALTRPIX'] = 22
del new_header['HISTORY']

In [ ]:
wcs_out = WCS(new_header)
new_header

SIMPLE  =                    T /Standard FITS                                   
BITPIX  =                  -32 / Short integer (16 bit)                         
NAXIS   =                    3                                                  
NAXIS1  =                  170                                                  
NAXIS2  =                  160                                                  
NAXIS3  =                  196                                                  
DATAMIN =       -8.0000000E+00                                                  
DATAMAX =        3.2000000E+01                                                  
BMAJ    =        2.5833334E-01                                                  
BMIN    =        2.5833334E-01                                                  
BPA     =        0.0000000E+00                                                  
                                                                                
BUNIT   = 'JY/BEAM '        

In [ ]:
hdu_mosaic = fits.PrimaryHDU(data=mosaic_cube, header=new_header)

In [ ]:
hipass_cube = sc.read(hdu_mosaic)
hipass_cube = hipass_cube.subcube(
    xlo=60 * u.deg,
    xhi=140 * u.deg,
    ylo=-13 * u.deg,
    yhi=2 * u.deg,
    zlo=-600 * u.km / u.s,
    zhi=600 * u.km / u.s,
)
hipass_cube

SpectralCube with shape=(92, 226, 1201) and unit=Jy / beam:
 n_x:   1201  type_x: RA---CAR  unit_x: deg    range:    60.020823 deg:  140.020827 deg
 n_y:    226  type_y: DEC--CAR  unit_y: deg    range:   -13.022828 deg:    1.977172 deg
 n_s:     92  type_s: VRAD      unit_s: m / s  range:  -600308.558 m / s:  600101.679 m / s

In [ ]:
# Find velocity axis (assume axis=2, adjust if necessary)
velocity_hel = hipass_cube.spectral_axis.to(u.km / u.s)
velocity_hel

<Quantity [-600.30855801, -587.11723672, -573.92591543, -560.73459413,
           -547.54327284, -534.35195155, -521.16063026, -507.96930897,
           -494.77798768, -481.58666639, -468.3953451 , -455.20402381,
           -442.01270251, -428.82138122, -415.63005993, -402.43873864,
           -389.24741735, -376.05609606, -362.86477477, -349.67345348,
           -336.48213219, -323.2908109 , -310.0994896 , -296.90816831,
           -283.71684702, -270.52552573, -257.33420444, -244.14288315,
           -230.95156186, -217.76024057, -204.56891928, -191.37759798,
           -178.18627669, -164.9949554 , -151.80363411, -138.61231282,
           -125.42099153, -112.22967024,  -99.03834895,  -85.84702766,
            -72.65570637,  -59.46438507,  -46.27306378,  -33.08174249,
            -19.8904212 ,   -6.69909991,    6.49222138,   19.68354267,
             32.87486396,   46.06618525,   59.25750655,   72.44882784,
             85.64014913,   98.83147042,  112.02279171,  125.214113  ,
      

In [ ]:
ra = 100 * u.deg
dec = 0 * u.deg
velocity_lsrk = []
for v in velocity_hel:
    sc = SkyCoord(ra, dec, radial_velocity=v, frame="fk5", pm_ra_cosdec=0*u.mas/u.yr, pm_dec=0*u.mas/u.yr, distance=10*u.kpc)
    velocity_lsrk.append(sc.lsrk.radial_velocity.value)
velocity_lsrk = np.array(velocity_lsrk) * u.km / u.s
velocity_lsrk

<Quantity [-617.41309137, -604.22177008, -591.03044879, -577.8391275 ,
           -564.64780621, -551.45648492, -538.26516362, -525.07384233,
           -511.88252104, -498.69119975, -485.49987846, -472.30855717,
           -459.11723588, -445.92591459, -432.7345933 , -419.54327201,
           -406.35195071, -393.16062942, -379.96930813, -366.77798684,
           -353.58666555, -340.39534426, -327.20402297, -314.01270168,
           -300.82138039, -287.63005909, -274.4387378 , -261.24741651,
           -248.05609522, -234.86477393, -221.67345264, -208.48213135,
           -195.29081006, -182.09948877, -168.90816748, -155.71684618,
           -142.52552489, -129.3342036 , -116.14288231, -102.95156102,
            -89.76023973,  -76.56891844,  -63.37759715,  -50.18627586,
            -36.99495456,  -23.80363327,  -10.61231198,    2.57900931,
             15.7703306 ,   28.96165189,   42.15297318,   55.34429447,
             68.53561576,   81.72693705,   94.91825835,  108.10957964,
      

In [ ]:
delta_v = []
for i in range(len(velocity_lsrk)):
    delta_v.append(velocity_lsrk[i].value - velocity_hel[i].value)
delta_v = np.mean(delta_v) * 1000
delta_v

np.float64(-17104.53336360639)

In [ ]:
# Now velocity_lsrk is your velocities in LSRK
cube_header = hipass_cube.header
cube_header['CRVAL3'] += delta_v

old_CRVAL1 = cube_header['CRVAL1']
new_CRVAL1 = 100
old_CRPIX1 = cube_header['CRPIX1']
CDELT1 = cube_header['CDELT1']
new_CRPIX1 = old_CRPIX1 - (old_CRVAL1 - new_CRVAL1) / CDELT1
cube_header['CRVAL1'] = new_CRVAL1
cube_header['CRPIX1'] = new_CRPIX1

# Update header
cube_header['SPECSYS'] = 'LSRK'
del cube_header['SLICE']
cube_header

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -64 / array data type                                
NAXIS   =                    3                                                  
NAXIS1  =                 1201                                                  
NAXIS2  =                  226                                                  
NAXIS3  =                   92                                                  
DATAMIN =       -8.0000000E+00                                                  
DATAMAX =        3.2000000E+01                                                  
BMAJ    =        2.5833334E-01                                                  
BMIN    =        2.5833334E-01                                                  
BPA     =        0.0000000E+00                                                  
BUNIT   = 'Jy beam-1'          / Brightness (pixel) unit                        
EPOCH   =   2.000000000000E+

In [ ]:
# Save to new FITS file
hipass_cube_hdu = fits.PrimaryHDU(data=hipass_cube.unmasked_data[:, :, :].value, header=cube_header)

hipass_cube_hdu.writeto('HIPASS_mosaic_cube.fits', overwrite=True)

In [ ]:
# Convert Jy/beam to K using brightness temperature equivalency
freq = hipass_cube.with_spectral_unit(u.Hz).spectral_axis
hipass_cube_K = hipass_cube.to(u.K, equivalencies=u.brightness_temperature(freq))

cube_header_K = cube_header.copy()
cube_header_K['BUNIT'] = "K"
cube_header_K

# Save to new FITS file
hipass_cube_K_hdu = fits.PrimaryHDU(data=hipass_cube_K.unmasked_data[:, :, :].value, header=cube_header_K)

hipass_cube_K_hdu.writeto('HIPASS_mosaic_cube_K.fits', overwrite=True)